# 16 — Supplementary solves: E12, E17-T3, E8–E10, E15-if-triggered (spec v0.13)

Consumes `spec/e_round_v13.json` (run 15 first). E12's three MAA sweeps dominate (~50 min
each); everything else is minutes. All artifacts resumable; live internet (WLS). Kernel
`R (y2y)`.

In [10]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd()")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
ctx <- pr_setup(mpath, PROJ)
ctx <- modifyList(ctx, pr_ingest(ctx))
ctx <- modifyList(ctx, pr_planning_units(ctx))
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
MANI <- read.csv(file.path(PROJ, "analyses/y2y/spec/manifest.csv"), stringsAsFactors = FALSE)
RUNS <- file.path(PROJ, "analyses/y2y/runs")
SC <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/scenarios_v2.json"))
S0W <- SC$S0_balanced$weights; S0T <- SC$S0_balanced$targets

build_form <- function(base_ctx, w, t, subdir_scratch) {
  actx <- pr_override(base_ctx, targets = t, feature_weight_multipliers = w,
                      results_subdir = subdir_scratch)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  actx
}
run_single <- function(base_ctx, w, t, out_rel, artifact = "run") {
  done <- file.path(PROJ, out_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s exists -- skipped\n", out_rel)); return(invisible(NULL)) }
  actx <- do.call(pr_override, c(list(base_ctx, targets = t, feature_weight_multipliers = w,
      results_dir = out_rel, results_subdir = artifact,
      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  invisible(NULL)
}
cat("ready | E15 trigger from E14:", isTRUE(ER$e14$trigger), "\n")

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
ready | E15 trigger from E14: TRUE 


In [11]:
# ---- E12: MAA bracketing sweeps (3 formulations, k=50, seeded) -----------------------------
for (fid in unlist(ER$e12$formulations)) {
  tif <- file.path(RUNS, fid, "maa_g05.tif")
  if (file.exists(tif)) { cat(sprintf("== %s maa exists -- skipped\n", fid)); next }
  cat(sprintf("\n===================== E12 %s =====================\n", fid))
  row <- MANI[MANI$formulation_id == fid, ]
  w <- jsonlite::fromJSON(row$weight_vector); t <- jsonlite::fromJSON(row$target_vector)
  actx <- build_form(ctx, w, t, "e12_build")
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = 1e-4)
  meta <- jsonlite::read_json(file.path(RUNS, fid, "formulation_meta.json"))
  stopifnot("anchor deviates from the frozen record" =
              abs(anchor$z - meta$anchor_objective) < 1e-3)
  gen <- maa_generate(cm, anchor, g = ER$e12$g, k = ER$e12$k,
                      seed = ER$e12$seeds[[fid]])
  dir.create(file.path(RUNS, fid), showWarnings = FALSE)
  layers <- lapply(seq_len(gen$k), function(i) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
    v[cm$pu_index] <- as.integer(gen$members[i, ]); terra::values(r) <- v; r })
  s <- terra::rast(layers); names(s) <- sprintf("maa_%02d", seq_len(gen$k))
  terra::writeRaster(s, tif, overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  write.csv(gen$certificates, file.path(RUNS, fid, "certificates_maa.csv"), row.names = FALSE)
  cat(sprintf("== %s MAA done\n", fid))
}

== s0_ssp585_theta5 maa exists -- skipped
== s2_ssp585_theta5 maa exists -- skipped
== s4_ssp585_theta3 maa exists -- skipped


In [12]:
# ---- E17-T3: leave-one-block-out counterfactual anchors at S0 (5 x ~1 min) -----------------
for (b in names(ER$e17_t3$blocks)) {
  w <- S0W
  for (f in unlist(ER$e17_t3$blocks[[b]])) w[[f]] <- 0
  run_single(ctx, w, S0T, file.path("analyses/y2y/runs/e17_t3", paste0(b, "_out")))
}
# EFG-out: zero every EFG multiplier (block list covers continuous blocks only)
efg_names <- ctx$layers$name[ctx$layers$role == "feature_efg"]
w <- S0W; for (f in efg_names) w[[f]] <- 0
run_single(ctx, w, S0T, "analyses/y2y/runs/e17_t3/efg_out")

   analyses/y2y/runs/e17_t3/core_habitat_out exists -- skipped
   analyses/y2y/runs/e17_t3/connectivity_out exists -- skipped
   analyses/y2y/runs/e17_t3/carbon_out exists -- skipped
   analyses/y2y/runs/e17_t3/biodiversity_out exists -- skipped
   analyses/y2y/runs/e17_t3/efg_out exists -- skipped


In [13]:
# ---- E8: m_soc weight x10 inertness (satiation) at S0 and S3 -------------------------------
for (fid in c("s0_ssp585_theta5", "s3_ssp585_theta5")) {
  row <- MANI[MANI$formulation_id == fid, ]
  w <- jsonlite::fromJSON(row$weight_vector); t <- jsonlite::fromJSON(row$target_vector)
  w$irrecoverable_carbon_m_soc <- w$irrecoverable_carbon_m_soc * 10
  run_single(ctx, w, t, file.path("analyses/y2y/runs/e8", paste0(substr(fid, 1, 2), "_carbonx10")))
}

   analyses/y2y/runs/e8/s0_carbonx10 exists -- skipped
   analyses/y2y/runs/e8/s3_carbonx10 exists -- skipped


In [14]:
# ---- E9: weights-only arm + log-carbon arm (layer patch) -----------------------------------
run_single(ctx, ER$e9$infweights$weights, ER$e9$infweights$targets,
           "analyses/y2y/runs/e9/infweights")
ctx_log <- pr_setup(mpath, PROJ)
ctx_log$layers$path[ctx_log$layers$name == "irrecoverable_carbon_m_soc"] <-
  ER$e9$logcarbon$layer_patch[["irrecoverable_carbon_m_soc"]]
ctx_log <- modifyList(ctx_log, pr_ingest(ctx_log))
ctx_log <- modifyList(ctx_log, pr_planning_units(ctx_log))
run_single(ctx_log, ER$e9$logcarbon$weights, ER$e9$logcarbon$targets,
           "analyses/y2y/runs/e9/logcarbon")

   analyses/y2y/runs/e9/infweights exists -- skipped
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
   analyses/y2y/runs/e9/logcarbon exists -- skipped


In [15]:
# ---- E10: theta capture-shift solves (theta10 t=0.121, theta3 t=0.552 at S0 shares) --------
run_single(ctx, ER$e10$theta10$weights, ER$e10$theta10$targets, "analyses/y2y/runs/e10/theta10")
run_single(ctx, ER$e10$theta3$weights,  ER$e10$theta3$targets,  "analyses/y2y/runs/e10/theta3")

   analyses/y2y/runs/e10/theta10 exists -- skipped
   analyses/y2y/runs/e10/theta3 exists -- skipped


In [16]:
# ---- E15 (CONDITIONAL on the E14 trigger): guardrailed-band MGA on S0 + S4 -----------------
if (!isTRUE(ER$e14$trigger)) {
  cat("E14 trigger NOT fired -- E15 skipped (pre-registered rule; see e_round_v13.json)\n")
} else {
  for (fid in c("s0_ssp585_theta5", "s4_ssp585_theta3")) {
    tif <- file.path(RUNS, fid, "mga_guard_g05.tif")
    if (file.exists(tif)) { cat(sprintf("== %s guard exists -- skipped\n", fid)); next }
    row <- MANI[MANI$formulation_id == fid, ]
    w <- jsonlite::fromJSON(row$weight_vector); t <- jsonlite::fromJSON(row$target_vector)
    actx <- build_form(ctx, w, t, "e15_build")
    cm <- mga_compile(actx)
    anchor <- mga_anchor(cm, opt_gap = 1e-4)
    blocks <- lapply(ER$e17_t3$blocks, unlist)
    gen <- mga_generate(cm, anchor, g = 0.05, k = 50,
                        floors = list(ctx = actx, blocks = blocks, g = 0.05))
    layers <- lapply(seq_len(gen$k), function(i) {
      r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
      v[cm$pu_index] <- as.integer(gen$members[i, ]); terra::values(r) <- v; r })
    s <- terra::rast(layers); names(s) <- sprintf("guard_%02d", seq_len(gen$k))
    terra::writeRaster(s, tif, overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                       gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    write.csv(gen$certificates, file.path(RUNS, fid, "certificates_guard.csv"), row.names = FALSE)
  }
}

== s0_ssp585_theta5 guard exists -- skipped
== s4_ssp585_theta3 guard exists -- skipped


In [17]:
# ---- E15b: per-VALUE floors (each of the 8 continuous values within 5% of anchor) ----------
# Ethan's escalation of E15: singleton "blocks" -- no individual value left >5% behind.
# EFGs stay the locked foundation (36/40 saturate; 40 extra rows would be decoration).
for (fid in c("s0_ssp585_theta5", "s4_ssp585_theta3")) {
  tif <- file.path(RUNS, fid, "mga_guardfeat_g05.tif")
  if (file.exists(tif)) { cat(sprintf("== %s guardfeat exists -- skipped\n", fid)); next }
  row <- MANI[MANI$formulation_id == fid, ]
  w <- jsonlite::fromJSON(row$weight_vector); t <- jsonlite::fromJSON(row$target_vector)
  actx <- build_form(ctx, w, t, "e15b_build")
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = 1e-4)
  cont_names <- actx$layers$name[actx$layers$role == "feature_continuous"]
  feat_blocks <- setNames(as.list(cont_names), cont_names)   # 8 singleton floors
  gen <- mga_generate(cm, anchor, g = 0.05, k = 50,
                      floors = list(ctx = actx, blocks = feat_blocks, g = 0.05))
  layers <- lapply(seq_len(gen$k), function(i) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
    v[cm$pu_index] <- as.integer(gen$members[i, ]); terra::values(r) <- v; r })
  s <- terra::rast(layers); names(s) <- sprintf("gf_%02d", seq_len(gen$k))
  terra::writeRaster(s, tif, overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  write.csv(gen$certificates, file.path(RUNS, fid, "certificates_guardfeat.csv"),
            row.names = FALSE)
}

  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464638, irrecoverable_carbon_biomass=0.19856, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  override results_subdir   -> e15b_build
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.332 | weight multipliers: climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464638, irrecoverable_carbon_biomass=0.19856, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  outputs  -> output_data/e15b_build
weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight climate_type_macrorefugia x1.5 -> 1.4600
  up-weight transboundary_connectivity x0.7 -> 0.6693
  up-weight climate_corridors x1.2 -> 1.1714
  up-weight irrecoverable_carbon_m_soc x0.5 -> 0.4646
  

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272962 cols (1272914 pu + 48 aux) x 49 rows | 191029 locked pu | modelsense min
anchor: objective 5.362811 (bound 5.362811, gap 0.00e+00) | 381,874 selected | 10 s
band wall appended: obj0 . x <= 5.630952  (g = 0.05 on z* = 5.362811)
  block floor human_modification anchor capture 0.2992 -> floor 0.2842
  block floor transboundary_connectivity anchor capture 0.3341 -> floor 0.3174
  block floor climate_corridors anchor capture 0.2603 -> floor 0.2472
  block floor climate_type_macrorefugia anchor capture 0.4521 -> floor 0.4295
  block floor irrecoverable_carbon_biomass anchor capture 0.3101 -> floor 0.2946
  block floor irrecoverable_carbon_m_soc anchor capture 0.3320 -> floor 0.3154
  block floor aoh_richness_mammals anchor capture 0.3267 -> floor 0.3103
  block floor aoh_richness_birds anchor capture 0.3300 -> floor 0.3135
g=0.05 iter 01/50: band 5.630952 (+5.00% of z*) OK | ham(anchor) 336,596 | 14 s
g=0.05 iter 02/50: band 5.630952 (+5.00% of z*) OK | ham(anchor) 257,276 

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.552 and 1)
││└•weights:    continuous values (between 0.025 and 1.436979)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272962 cols (1272914 pu + 48 aux) x 49 rows | 191029 locked pu | modelsense min
anchor: objective 5.014282 (bound 5.014282, gap 0.00e+00) | 381,875 selected | 12 s
band wall appended: obj0 . x <= 5.264996  (g = 0.05 on z* = 5.014282)
  block floor human_modification anchor capture 0.3019 -> floor 0.2868
  block floor transboundary_connectivity anchor capture 0.3214 -> floor 0.3054
  block floor climate_corridors anchor capture 0.2840 -> floor 0.2698
  block floor climate_type_macrorefugia anchor capture 0.4176 -> floor 0.3967
  block floor irrecoverable_carbon_biomass anchor capture 0.3288 -> floor 0.3123
  block floor irrecoverable_carbon_m_soc anchor capture 0.5520 -> floor 0.5244
  block floor aoh_richness_mammals anchor capture 0.3098 -> floor 0.2943
  block floor aoh_richness_birds anchor capture 0.3082 -> floor 0.2928
g=0.05 iter 01/50: band 5.264996 (+5.00% of z*) OK | ham(anchor) 306,214 | 11 s
g=0.05 iter 02/50: band 5.264996 (+5.00% of z*) OK | ham(anchor) 223,891 

In [18]:
# ---- round summary -------------------------------------------------------------------------
cat("E12 MAA:", paste(sapply(unlist(ER$e12$formulations), function(f)
  if (file.exists(file.path(RUNS, f, "maa_g05.tif"))) "done" else "TODO"), collapse = " "), "\n")
for (d in c("e17_t3/biodiversity_out", "e17_t3/efg_out", "e8/s0_carbonx10",
            "e9/infweights", "e9/logcarbon", "e10/theta10", "e10/theta3"))
  cat(sprintf("%-24s %s\n", d,
              if (file.exists(file.path(RUNS, d, "run/run_summary.json"))) "done" else "TODO"))
cat("\nnext: analyses/y2y/17_e_round_analysis.ipynb\n")

E12 MAA: done done done 
e17_t3/biodiversity_out  done
e17_t3/efg_out           done
e8/s0_carbonx10          done
e9/infweights            done
e9/logcarbon             done
e10/theta10              done
e10/theta3               done

next: analyses/y2y/17_e_round_analysis.ipynb
